In [ ]:
# add default values for parameters here
domains = None
upstream = ['classify']
product = None

# Filter host

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import taxoniq
import tqdm

In [ ]:
kraken2_out = pd.read_csv(upstream['classify']['kraken_out'], sep="\t", header=None, index_col=None, names=['is_classified', 'seq_id',  'taxonid', '_1', '_2'])
kraken2_out[:10]


## Figure 1: Classified value counts

In [ ]:
kraken2_out.value_counts(subset=['is_classified']).reset_index()

In [ ]:
g = sns.catplot(kraken2_out.value_counts(subset=['is_classified']).reset_index(), x='is_classified', y='count', kind='bar')
# g.axes[0, 0].set_yscale('log')

## Figure 2: Top species among classification

In [ ]:
kraken2_classified = kraken2_out.query('is_classified == "C"').copy()
kraken2_classified_bycount = kraken2_classified.value_counts(['taxonid']).reset_index()
kraken2_classified_bycount['position'] = np.arange(len(kraken2_classified_bycount))

In [ ]:
g = sns.relplot(x='position', y='count', data=kraken2_classified_bycount, kind='line', height=2.0, aspect=3.0)
g.axes[0,0].set_yscale('log')

In [ ]:
kraken2_classified_bycount_top = kraken2_classified_bycount[:50].copy()
kraken2_classified_bycount_top['logcount'] = kraken2_classified_bycount_top['count'].apply(np.log10)
kraken2_classified_bycount_top['scientific name'] = kraken2_classified_bycount_top['taxonid'].apply(
    lambda x: taxoniq.Taxon(x).scientific_name
)

In [ ]:
sns.catplot(y='scientific name', x='logcount', data=kraken2_classified_bycount_top, height=8.0, aspect=0.5, kind='bar')

## Figure 3: Hits by domain

In [ ]:
def obtain_domain(taxonid):
    try:
        return taxoniq.Taxon(taxonid).ranked_lineage[-1].scientific_name
    except IndexError:
        return "unknown"
    except KeyError:
        return "unknown"

def obtain_scientific_name(taxonid):
    try:
        return taxoniq.Taxon(taxonid).scientific_name
    except IndexError:
        return "unknown"
    except KeyError:
        return "unknown"


In [ ]:
kraken2_classified_bycount['domain'] = kraken2_classified_bycount['taxonid'].apply(obtain_domain)
kraken2_classified_bycount['scientific_name'] = kraken2_classified_bycount['taxonid'].apply(obtain_scientific_name)

In [ ]:
g = sns.catplot(
    data=kraken2_classified_bycount.value_counts(subset=['domain']).reset_index(),
    x='domain', y='count', kind='bar', height=3.5
)
g.set_xticklabels(rotation=45)

## Figure 4: Reads by domain

In [ ]:
g = sns.catplot(
    data=kraken2_classified_bycount[['domain', 'count']].groupby('domain', as_index=False).sum(),
    x='domain', y='count', kind='bar', height=3.5
)
g.set_xticklabels(rotation=45)

## Output

In [ ]:
taxonomy = []
for item in tqdm.tqdm(kraken2_out['taxonid'].unique()):
    try:
        t = taxoniq.Taxon(item)
        domain = t.ranked_lineage[-1].scientific_name
        kingdom = t.ranked_lineage[-2].scientific_name
    except KeyError:
        taxonomy.append(dict(taxonid=item, species_name="unknown", domain="unknown", kingdom="unknown"))
        continue
    except IndexError:
        taxonomy.append(dict(taxonid=item, species_name=t.scientific_name, domain="unknown", kingdom="unknown"))
        continue
    taxonomy.append(
        dict(
            taxonid=item, 
            species_name=t.scientific_name,
            domain=domain,
            kingdom=kingdom
        )
    )
taxonomy = pd.DataFrame.from_records(taxonomy)

In [ ]:
kraken2_out = pd.merge(kraken2_out, taxonomy, on='taxonid')
kraken2_out[
    kraken2_out['domain'].isin(domains)
][['seq_id']].to_csv(product['csv'], sep="\t", index=None, header=None)